### Re-ranking

In [1]:
import os
import sys
import joblib

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

dir = "artifacts/ngcf"
prefix = "ngcf_k_all"
eval_df = joblib.load(os.path.join(dir, f"{prefix}_eval_df.pkl"))
user_dps_df = joblib.load(os.path.join(dir, f"user_dps_df.pkl"))
feature_engineer = joblib.load(os.path.join(dir, f"feature_engineer.pkl"))

/home/adam/R11_Bai/DPRecSys/.venv/lib/python3.11/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


### MMR

In [2]:
from post_processing.mmr import MMR

mmr_reranker = MMR(
    eval_df=eval_df,
    feature_engineer=feature_engineer,
    m=100,
)


Seed set to 42


In [3]:
mmr_result_df = mmr_reranker.rerank(
    top_k=20,
    theta=0.5,
    random_state=42,
)
mmr_result_df.head()

Preparing input DataFrame for MMR...
candidate item pool size: 100
exploded 2064
extracting item features...
merging features...
interaction data count before merging: 206400
interaction data count after merging: 206400
done!
Reranking items for each user...


Reranking users: 100%|██████████| 2064/2064 [00:31<00:00, 66.34it/s]

Reranked DataFrame shape: (2064, 2)
Final Reranked DataFrame shape: (2064, 4)


,user,rec_items,asis_rec_items,gt_items
0,75,"[1333, 475, 1199, 50, 4993, 2571, 4995, 1270, ...","[1333, 318, 475, 50, 6711, 2858, 2571, 593, 86...","[2058, 163, 2490, 45722, 1233, 110, 2959, 2571]"
1,78,"[50, 4993, 1333, 47, 5669, 293, 2858, 1199, 11...","[50, 1333, 47, 7153, 6539, 4993, 1036, 5952, 1...","[5881, 37729, 44191, 4119, 6993, 8400, 50872]"
2,127,"[40966, 34321, 26554, 41997, 44788, 46855, 630...","[40966, 26554, 44788, 30812, 6305, 50794, 4199...","[45726, 6958]"
3,170,"[4973, 152, 1061, 5922, 6385, 2716, 1201, 5787...","[4973, 2396, 2716, 152, 1061, 5922, 5787, 3379...","[4963, 1222, 3949, 4011, 2542, 8874, 44191, 45..."
4,175,"[1270, 475, 4993, 293, 1333, 39183, 1036, 3996...","[1270, 4993, 6711, 475, 1036, 2858, 33493, 133...","[1921, 5995, 1913, 1419, 4927, 50068, 7700, 17..."


In [4]:
from common.eval import Evaluator
evaluator = Evaluator()

mmr_reranked_score_df = evaluator.evaluate(mmr_result_df, K=5)
mmr_reranked_score_df = evaluator.evaluate(mmr_reranked_score_df, K=10)
mmr_reranked_score_df = evaluator.evaluate(mmr_reranked_score_df, K=20)
mmr_reranked_score_df.describe()

Seed set to 42


,user,ndcg@5,recall@5,precision@5,ndcg@10,recall@10,precision@10,ndcg@20,recall@20,precision@20
count,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000
mean,35564.367733,0.117274,0.012783,0.042054,0.146947,0.023050,0.037355,0.174922,0.042341,0.034520
std,20797.975208,0.274283,0.043223,0.099123,0.269620,0.060969,0.070487,0.253263,0.087538,0.054301
min,75.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,17798.500000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,35054.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,53331.000000,0.000000,0.000000,0.000000,0.301030,0.020254,0.100000,0.315465,0.057143,0.050000
max,71534.000000,1.000000,1.000000,0.800000,1.000000,1.000000,0.600000,1.000000,1.000000,0.350000


In [5]:
reranked_user_dpms_df = evaluator.evaluate_dpms_at_k(
    eval_df=mmr_result_df,
    feature_engineer=feature_engineer,
    ground_truth_dps_df=user_dps_df,
    k=10,
    actor_k=5,
    rare_threshold=5,
)

reranked_user_dpms_df.describe()

candidate item pool size: 10
exploded 2064
extracting item features...
merging features...
interaction data count before merging: 20640
interaction data count after merging: 20640
done!
Transformed: Re-index user/item mapping
Transformed: Encoded idx for actorID
Transformed: Encoded idx for country
Transformed: Encoded idx for directorID
Transformed: Encoded idx for genre
encoded 2064


Calculating user diversity preference scale: 100%|██████████| 2064/2064 [00:02<00:00, 858.98it/s]


combined_df 2064
Calculating DPMS for each feature...


,userID,actorID_dpms,country_dpms,directorID_dpms,genre_dpms,avg_dpms
count,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000
mean,1031.500000,0.079430,0.852285,0.086505,0.825973,0.461048
std,595.969798,0.041921,0.091922,0.072599,0.085963,0.042816
min,0.000000,0.000000,0.356122,0.000000,0.386859,0.294032
25%,515.750000,0.048327,0.799370,0.026487,0.784359,0.433067
50%,1031.500000,0.077544,0.868124,0.077005,0.843134,0.463300
75%,1547.250000,0.106958,0.924966,0.132428,0.887884,0.491215
max,2063.000000,0.243009,0.995940,0.398429,0.970755,0.593382


In [6]:
# ILS@10
ils_df = evaluator.evaluate_ils_at_k(mmr_result_df, k=10)
ils_df.describe()

,user,ILS@10
count,2064.000000,2064.000000
mean,35564.367733,0.095615
std,20797.975208,0.026847
min,75.000000,0.036257
25%,17798.500000,0.076594
50%,35054.000000,0.090535
75%,53331.000000,0.110853
max,71534.000000,0.234074


### DPA-RS

In [7]:
from post_processing.dpa_rs import DPA_RS

reranker = DPA_RS(
    eval_df=eval_df,
    feature_engineer=feature_engineer,
    m=100,
    ground_truth_dps_df=user_dps_df
)


Seed set to 42


In [ ]:
dpa_result_df = reranker.rerank(
    top_k=20,
    max_iter=100,
    random_state=42,
)
dpa_result_df.head()

Preparing input DataFrame for DPA-RS...
candidate item pool size: 100
exploded 2064
extracting item features...
merging features...
interaction data count before merging: 206400
interaction data count after merging: 206400
done!
Transformed: Re-index user/item mapping
Transformed: Encoded idx for actorID
Transformed: Encoded idx for country
Transformed: Encoded idx for directorID
Transformed: Encoded idx for genre
encoded 2064
Combined DataFrame shape: (206400, 16)
Reranking items for each user...


Reranking users:  62%|██████▏   | 1287/2064 [06:47<03:38,  3.55it/s]/home/adam/R11_Bai/DPRecSys/.venv/lib/python3.11/site-packages/cvxpy/problems/problem.py:1510: UserWarning: Solution may be inaccurate. Try another solver, adjusting the solver settings, or solve with verbose=True for more information.
  warnings.warn(
Reranking users:  65%|██████▌   | 1345/2064 [07:05<03:20,  3.58it/s]

In [ ]:
from common.eval import Evaluator
evaluator = Evaluator()

reranked_score_df = evaluator.evaluate(dpa_result_df, K=5)
reranked_score_df = evaluator.evaluate(reranked_score_df, K=10)
reranked_score_df = evaluator.evaluate(reranked_score_df, K=20)
reranked_score_df.describe()

Seed set to 42


,user,ndcg@5,recall@5,precision@5,ndcg@10,recall@10,precision@10,ndcg@20,recall@20,precision@20
count,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000
mean,35564.367733,0.131062,0.016409,0.051647,0.171711,0.030401,0.048983,0.204450,0.055757,0.044695
std,20797.975208,0.280640,0.048346,0.115444,0.276272,0.066522,0.085831,0.258575,0.093505,0.065937
min,75.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,17798.500000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,35054.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,53331.000000,0.000000,0.000000,0.000000,0.333333,0.038462,0.100000,0.357186,0.083333,0.050000
max,71534.000000,1.000000,0.666667,0.800000,1.000000,0.750000,0.700000,1.000000,1.000000,0.500000


In [ ]:
reranked_user_dpms_df = evaluator.evaluate_dpms_at_k(
    eval_df=dpa_result_df,
    feature_engineer=feature_engineer,
    ground_truth_dps_df=user_dps_df,
    k=10,
    actor_k=5,
    rare_threshold=5,
)

reranked_user_dpms_df.describe()

candidate item pool size: 10
exploded 2064
extracting item features...
merging features...
interaction data count before merging: 20640
interaction data count after merging: 20640
done!
Transformed: Re-index user/item mapping
Transformed: Encoded idx for actorID
Transformed: Encoded idx for country
Transformed: Encoded idx for directorID
Transformed: Encoded idx for genre
encoded 2064


Calculating user diversity preference scale: 100%|██████████| 2064/2064 [00:02<00:00, 845.52it/s]


combined_df 2064
Calculating DPMS for each feature...


,userID,actorID_dpms,country_dpms,directorID_dpms,genre_dpms,avg_dpms
count,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000
mean,1031.500000,0.195792,0.977313,0.398290,0.922210,0.623401
std,595.969798,0.063160,0.025039,0.159864,0.034683,0.047289
min,0.000000,0.000000,0.738622,0.000000,0.715724,0.465622
25%,515.750000,0.156412,0.971318,0.302336,0.904272,0.593779
50%,1031.500000,0.195160,0.984627,0.411996,0.928441,0.627743
75%,1547.250000,0.235357,0.992131,0.507284,0.946415,0.655626
max,2063.000000,0.603120,1.000000,0.918092,0.989566,0.776087


In [ ]:
# ILS@10
ils_df = evaluator.evaluate_ils_at_k(dpa_result_df, k=10)
ils_df.describe()